In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from itertools import chain

In [0]:
def circuits_specific(df):
    country_map = {
        "USA": "United States",
        "UK": "United Kingdom",
        "UAE": "United Arab Emirates"
    }
    map_expr = F.create_map([F.lit(x) for x in chain(*country_map.items())])
    df = df.withColumn(
        "country",
        F.coalesce(map_expr[F.col("country")], F.col("country"))
    )
    return df

def constructor_results_specific(df):
    # Status mapping: D -> Disqualified
    df = df.withColumn("status", F.when(F.col("status") == "D", "Disqualified").otherwise(F.col("status")))
    return df

def drivers_specific(df):
    # Trim names and handle DOB/Number
    df = df.withColumn("forename", F.trim(F.col("forename"))) \
           .withColumn("surname", F.trim(F.col("surname")))
    return df

def races_specific(df):
    df = df.drop("year")
    return df

def results_specific(df):
    df = df.withColumn("fastestLapSpeed", F.col("fastestLapSpeed").cast(DoubleType()))
    return df

def season_specific(df):
    df = df.withColumn("year", F.col("year").cast(IntegerType()))
    return df

def red_flags_specific(df):
    # Define the mapping
    mapping = {
        "R": "Resumed",
        "S": "Standing Start",
        "Y": "Rolling Start",
        "N": "Not Resumed"
    }
    map_expr = F.create_map([F.lit(x) for x in chain(*mapping.items())])
    df = df.withColumn(
        "resumed", 
        F.coalesce(map_expr[F.col("resumed")], F.col("resumed"))
    )
    return df

def safety_car_specific(df):
    df = df.withColumn("cause", 
        F.when(F.col("cause") == "multiple accidents", F.col("cause")) # Keep as is
            .when(F.col("cause").rlike("(?i)^accident.*"), "accident")        # Group all accident types
            .when(F.col("cause").rlike("(?i)^stranded car.*"), "stranded car") # Group all stranded types
            .otherwise(F.col("cause"))                                        # Keep everything else
    )
    return df

In [0]:
def apply_specific_silver_transforms(df, table_name ):
    # Dictionary mapping table names to their specific functions
    f1_transform_map = {
        "race_data_circuits": circuits_specific,
        "race_data_constructor_results": constructor_results_specific,
        "race_data_drivers": drivers_specific,
        "race_data_races": races_specific,
        "race_data_results": results_specific,
        "race_data_seasons": season_specific,
        "race_events_red_flags": red_flags_specific,
        "race_events_safety_car": safety_car_specific
    }
    
    # Look up the function in the dictionary
    if table_name in f1_transform_map:
        # Run the specific function and update df
        df = f1_transform_map[table_name](df)
        
    return df